# Notebook 03 - Feature Engineering (Customer-Level)

**Input:** `data/interim/transactions_customer_level.parquet` (~800K rows, 5,861 customers)

**Output:** `data/processed/customer_features.parquet` - one row per customer with ~25 features ready for segmentation, CLV, and churn modeling

## Why this notebook is the most important one

In ML, you'll often hear:*"better features beat better models."* Spending an extra day here saves a week of trying to squeeze accuracy out of model tuning

## The critical concept: snapshot date and temporal splits

**Wrong way:** Compute all features over the entire dateset, then try to predict "will customer churn". This **leaks the future into the features** - your features know what hasn't happened yet from the model's perspective.

**Right way:** Pick a `snapshot_date`. Compute features using ONLY data up to that date. Define the target using data AFTER that date. The model learns to predict the unknown future from the known past - exactly how it will operate when deployed.

Our data ends `2011-12-09`. We'll use:
- **Feature window:** all data up to `2011-09-09` (snapshot date)
- **Target window:** `2011-09-10` to `2011-12-09` (next 90 days)

Customers who first purchased AFTER the snapshot date can't be features-engineered (no history). We exclude them here.

## Feature groups we'll build

1. **RFM** — Recency, Frequency, Monetary (the foundational triad)
2. **Behavioral** — AOV, basket size, days between orders, consistency
3. **Temporal/trend** — recent vs. historical activity, momentum
4. **Product mix** — diversity, category breadth (proxied by stock-code count)
5. **Geography** — country, is_uk flag
6. **Return behavior** — return rate, return count (from cancellations table)
7. **Targets** — for downstream supervised tasks:
   - `target_revenue_90d` (regression target for CLV)
   - `target_purchased_90d` (binary, classification target for churn)

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_style('whitegrid')

INTERIM_DIR = Path('../data/interim')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load data and efine the tme split

We'll define snapshot date by working backward from the data's max date. Using **90 days** as our target window is a defensible choice <br>
for an e-commerce business - long enough to capture re-purchase behavior, short enough to be actionable.

In [2]:
df = pd.read_parquet(INTERIM_DIR / 'transactions_customer_level.parquet')
cancellations = pd.read_parquet(INTERIM_DIR / 'cancellations.parquet')

data_max_date = df['InvoiceDate'].max()
TARGET_WINDOW_DAYS = 90
snapshot_date = data_max_date - timedelta(days=TARGET_WINDOW_DAYS)

print(f'Data spans:     {df['InvoiceDate'].min()} -> {data_max_date}')
print(f'Snapshot date:  {snapshot_date}')
print(f'Feature window: start -> {snapshot_date}')
print(f'Target window:  {snapshot_date} -> {data_max_date} ({TARGET_WINDOW_DAYS} days)')


Data spans:     2009-12-01 07:45:00 -> 2011-12-09 12:50:00
Snapshot date:  2011-09-10 12:50:00
Feature window: start -> 2011-09-10 12:50:00
Target window:  2011-09-10 12:50:00 -> 2011-12-09 12:50:00 (90 days)


In [4]:
# Split Data
feat_df = df[df['InvoiceDate'] <= snapshot_date].copy()
target_df = df[df['InvoiceDate'] > snapshot_date].copy()

print(f'Feature window rows: {len(feat_df):,}')
print(f'Target window rows: {len(target_df):,}')
print()
print(f'Customers visible in feature window: {feat_df['CustomerID'].nunique():,}')
print(f'Customers visible in target window: {target_df['CustomerID'].nunique():,}')
print(f'Customers in both (will become positives for churn target): '
      f'{len(set(feat_df['CustomerID']) & set(target_df['CustomerID'])):,}')

Feature window rows: 641,705
Target window rows: 160,932

Customers visible in feature window: 5,256
Customers visible in target window: 2,885
Customers in both (will become positives for churn target): 2,289


## 2. Initialize the feature table

Start with one row per customer who exists in the feature window. We'll join features onto this skeleton as we build them.

In [5]:
features = pd.DataFrame({'CustomerID': feat_df['CustomerID'].unique()}).sort_values('CustomerID').reset_index(drop=True)
print(f'Customer skeleton: {len(features):,} rows')
features.head()

Customer skeleton: 5,256 rows


,CustomerID
0,12346.0
1,12347.0
2,12348.0
3,12349.0
4,12350.0


## 3. RFM features

**Recency** - Days since last purchase as of snapshot date. Lower is better (more recent = more likely to repeat)

**Frequency** - Number of distinct invoices in the feature window. Higher = more engaged.

**Monetary** - Total revenue in feature window. The classic measure of customer value.

In [9]:
rfm = feat_df.groupby('CustomerID').agg(
    last_purchase_date=('InvoiceDate', 'max'),
    first_purchase_date=('InvoiceDate', 'min'),
    frequency=('Invoice', 'nunique'),
    monetary=('Revenue', 'sum'),
    total_units=('Quantity', 'sum'),
    total_line_items=('Invoice', 'count')
).reset_index()

rfm['recency_days'] = (snapshot_date - rfm['last_purchase_date']).dt.days
rfm['tenure_days'] = (rfm['last_purchase_date'] - rfm['first_purchase_date']).dt.days
rfm['customer_age_days'] = (snapshot_date - rfm['first_purchase_date']).dt.days

features = features.merge(rfm, on='CustomerID', how='left')
features[['CustomerID', 'recency_days', 'frequency', 'monetary', 'tenure_days']].describe()

,recency_days,frequency,monetary,tenure_days
count,"5,256.00","5,256.00","5,256.00","5,256.00"
mean,206.36,5.71,"2,672.38",224.43
std,174.35,11.22,"12,407.43",218.60
min,0.00,1.00,1.55,0.00
25%,49.00,1.00,322.29,0.00
50%,163.00,3.00,798.00,171.00
75%,324.00,6.00,"2,106.93",418.00
max,648.00,284.00,"484,615.10",646.00
